# GomezAgent — Connect-4
## Análisis del Agente

---

Este notebook documenta el diseño y análisis empírico del agente de Juan Gomez, un agente que combina
reglas de prioridad determinísticas con *Monte Carlo Tree Search* (MCTS / UCT) para jugar Connect-4.

**Contenido:**
1. Descripción del algoritmo
2. Experimento 1 – Desempeño contra el jugador aleatorio (Rojo y Amarillo)
3. Experimento 2 – Autodesempeño (Gomez vs Gomez)
4. Experimento 3 – Sensibilidad al presupuesto de tiempo (`TIME_BUDGET`)
5. Experimento 4 – Ablación de componentes
6. Conclusiones y propuestas de mejora

> **Tiempo estimado de ejecución total:** ~10–15 min.


## 1. El Agente: GomezAgent

### Idea principal
GomezAgent combina dos capas de decisión que se ejecutan en orden de prioridad:

| Prioridad | Capa | Descripción |
|-----------|------|-------------|
| 1° | **Reglas determinísticas** | Si puedo ganar ahora, gano. Si el oponente puede ganar ahora, bloqueo. |
| 2° | **MCTS / UCT** | Búsqueda en árbol con simulaciones aleatorias para elegir el mejor movimiento a largo plazo. |

### ¿Qué hace diferente a este agente?

- **Tabla-Q persistente entre partidas** (`_q_table`): al terminar cada juego, las estadísticas del
  árbol MCTS se guardan en un diccionario de clase. En la siguiente partida, esas estadísticas se usan
  como *prior* en el UCT, acelerando la convergencia desde la primera simulación.
- **Hash Zobrist incremental** (`_ZOB`): en lugar de recalcular el hash del tablero desde cero en
  cada expansión, se actualiza con XOR en O(1).
- **Inferencia de color automática**: el agente deduce si juega como Rojo (-1) o Amarillo (+1)
  contando las piezas en el tablero, sin necesidad de parámetro externo.

### Parámetro de configuración
```python
GomezAgent.TIME_BUDGET = 5.0  # segundos por movimiento
```
El `TIME_BUDGET` es la variable numérica principal que controla la calidad vs. la velocidad del agente.


In [ ]:
import sys, os, random, time, math, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# Buscar la raíz del proyecto (donde está la carpeta connect4/)
def find_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.isdir(os.path.join(path, 'connect4')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError('No se encontró la raíz del torneo (carpeta connect4/).')

ROOT = find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from connect4.connect_state import ConnectState
from connect4.policy import Policy

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print(f'Raiz del proyecto: {ROOT}')
print('Imports OK')


## Código del agente (copia completa)

La siguiente celda reproduce el agente tal como aparece en `juanfe/policy.py`.


In [ ]:
# Helpers de tablero
ROWS, COLS = 6, 7

# Tabla Zobrist para hash incremental
_ZOB     = np.random.default_rng(42).integers(0, 2**63, (ROWS, COLS, 3), dtype=np.uint64)
_ZOB_FLAT = _ZOB.reshape(ROWS * COLS, 3)

def _board_hash(board):
    idx = (board + 1).ravel()
    return int(np.bitwise_xor.reduce(_ZOB_FLAT[np.arange(ROWS * COLS), idx]))

def _valid_moves(board):
    return [c for c in range(COLS) if board[0, c] == 0]

def _drop_row(board, col):
    for r in range(ROWS - 1, -1, -1):
        if board[r, col] == 0:
            return r
    return -1

def _drop(board, col, player):
    b = board.copy()
    b[_drop_row(b, col), col] = player
    return b

def _winner(board):
    for r in range(ROWS):
        for c in range(COLS):
            p = board[r, c]
            if p == 0:
                continue
            if c+3 < COLS and board[r,c+1]==p and board[r,c+2]==p and board[r,c+3]==p: return p
            if r+3 < ROWS and board[r+1,c]==p and board[r+2,c]==p and board[r+3,c]==p: return p
            if r+3<ROWS and c+3<COLS and board[r+1,c+1]==p and board[r+2,c+2]==p and board[r+3,c+3]==p: return p
            if r+3<ROWS and c-3>=0 and board[r+1,c-1]==p and board[r+2,c-2]==p and board[r+3,c-3]==p: return p
    return 0

def _wins_now(board, col, player):
    row = _drop_row(board, col)
    if row < 0: return False
    b = board.copy()
    b[row, col] = player
    return _winner(b) == player

def _rollout(board, player):
    b, p = board.copy(), player
    while True:
        w = _winner(b)
        if w != 0: return w
        moves = _valid_moves(b)
        if not moves: return 0
        b = _drop(b, random.choice(moves), p)
        p = -p

# Nodo MCTS
class _Node:
    __slots__ = ('board','player','move','parent','children','untried','wins','visits','bhash')

    def __init__(self, board, player, move=None, parent=None, bhash=None):
        self.board   = board
        self.player  = player
        self.move    = move
        self.parent  = parent
        self.children = []
        self.untried  = _valid_moves(board)
        random.shuffle(self.untried)
        self.wins    = 0.0
        self.visits  = 0
        self.bhash   = bhash if bhash is not None else _board_hash(board)

    @property
    def is_terminal(self):      return _winner(self.board) != 0 or not _valid_moves(self.board)
    @property
    def is_fully_expanded(self): return not self.untried

    def best_uct(self, c=1.414, q_table=None):
        log_n = math.log(self.visits)
        def uct(ch):
            score = ch.wins / ch.visits + c * math.sqrt(log_n / ch.visits)
            if q_table:
                entry = q_table.get((self.bhash, ch.move))
                if entry and entry[1]:
                    score += 0.25 * (entry[0] / entry[1]) / (1.0 + ch.visits)
            return score
        return max(self.children, key=uct)

    def expand(self):
        col = self.untried.pop()
        row = _drop_row(self.board, col)
        p_idx    = 0 if self.player == -1 else 2
        new_hash = self.bhash ^ int(_ZOB[row,col,1]) ^ int(_ZOB[row,col,p_idx])
        new_board = self.board.copy()
        new_board[row, col] = self.player
        child = _Node(new_board, -self.player, move=col, parent=self, bhash=new_hash)
        self.children.append(child)
        return child

# Q-table flush
_Q_MAX       = 300_000
_Q_MIN_VISITS = 10
_Q_FLUSH_SEC  = 0.3

def _flush_to_q(root, q_table):
    cutoff = time.time() + _Q_FLUSH_SEC
    stack = [root]
    while stack and time.time() < cutoff:
        node = stack.pop()
        if node.visits >= _Q_MIN_VISITS and node.parent is not None:
            key = (node.parent.bhash, node.move)
            if key in q_table or len(q_table) < _Q_MAX:
                if key not in q_table:
                    q_table[key] = [0.0, 0]
                q_table[key][0] += node.wins
                q_table[key][1] += node.visits
        for child in node.children:
            if child.visits >= _Q_MIN_VISITS:
                stack.append(child)

# Motor MCTS
def _mcts(board, player, budget, q_table):
    root     = _Node(board, player)
    deadline = time.time() + budget
    while time.time() < deadline:
        node = root
        while node.is_fully_expanded and not node.is_terminal:
            node = node.best_uct(q_table=q_table)
        if not node.is_terminal and not node.is_fully_expanded:
            node = node.expand()
        winner = _rollout(node.board, node.player)
        cur = node
        while cur is not None:
            cur.visits += 1
            if cur.parent is not None:
                mover = cur.parent.player
                if winner == mover:  cur.wins += 1.0
                elif winner == 0:    cur.wins += 0.5
            cur = cur.parent
    _flush_to_q(root, q_table)
    if not root.children:
        moves = _valid_moves(board)
        return min(moves, key=lambda c: abs(c - 3))
    return max(root.children, key=lambda ch: ch.visits).move

# Agente principal
class GomezAgent(Policy):
    TIME_BUDGET = 5.0
    _MAX_BUDGET = 0.5
    _q_table    = {}

    def __init__(self):
        try: super().__init__()
        except Exception: pass
        self.my_color = None
        self._budget  = self.__class__.TIME_BUDGET

    def mount(self, timeout=None):
        self.my_color = None
        if timeout is not None:
            self._budget = max(0.5, timeout * 0.80 - _Q_FLUSH_SEC)
        else:
            self._budget = self.__class__.TIME_BUDGET

    def _infer_color(self, board):
        return -1 if int(np.sum(board == -1)) == int(np.sum(board == 1)) else 1

    def act(self, board: np.ndarray) -> int:
        if self.my_color is None:
            self.my_color = self._infer_color(board)
        me    = self.my_color
        moves = _valid_moves(board)
        if len(moves) == 1: return moves[0]
        for col in sorted(moves, key=lambda c: abs(c - 3)):
            if _wins_now(board, col, me):  return col
        for col in sorted(moves, key=lambda c: abs(c - 3)):
            if _wins_now(board, col, -me): return col
        return _mcts(board, me, self._budget, self.__class__._q_table)

print('GomezAgent definido correctamente.')


## Agentes auxiliares y función de juego

Para los experimentos necesitamos:
- **AgenteAleatorio** — elige columna libre al azar (piso de referencia).
- **AgenteSoloPrioridad** — solo las reglas de prioridad (ganar/bloquear), sin MCTS.
- **AgenteMCTSSinQ** — MCTS completo pero **sin** la tabla-Q: no aprende entre partidas.


In [ ]:
class AgenteAleatorio(Policy):
    """Elige una columna libre al azar."""
    def mount(self): pass
    def act(self, board):
        moves = [c for c in range(COLS) if board[0, c] == 0]
        return random.choice(moves)


class AgenteSoloPrioridad(Policy):
    """Aplica las reglas de prioridad (ganar/bloquear) y en otro caso juega aleatorio."""
    def __init__(self):
        try: super().__init__()
        except Exception: pass
        self.my_color = None

    def mount(self, timeout=None):
        self.my_color = None

    def act(self, board):
        if self.my_color is None:
            self.my_color = -1 if int(np.sum(board==-1))==int(np.sum(board==1)) else 1
        me    = self.my_color
        moves = _valid_moves(board)
        if len(moves) == 1: return moves[0]
        for col in sorted(moves, key=lambda c: abs(c - 3)):
            if _wins_now(board, col, me):  return col
        for col in sorted(moves, key=lambda c: abs(c - 3)):
            if _wins_now(board, col, -me): return col
        return random.choice(moves)  


class AgenteMCTSSinQ(Policy):
    """MCTS completo pero Q-table vacía en cada llamada (sin aprendizaje cross-game)."""
    TIME_BUDGET = 0.5

    def __init__(self):
        try: super().__init__()
        except Exception: pass
        self.my_color = None
        self._budget  = self.__class__.TIME_BUDGET

    def mount(self, timeout=None):
        self.my_color = None
        self._budget  = self.__class__.TIME_BUDGET

    def act(self, board):
        if self.my_color is None:
            self.my_color = -1 if int(np.sum(board==-1))==int(np.sum(board==1)) else 1
        me    = self.my_color
        moves = _valid_moves(board)
        if len(moves) == 1: return moves[0]
        for col in sorted(moves, key=lambda c: abs(c - 3)):
            if _wins_now(board, col, me):  return col
        for col in sorted(moves, key=lambda c: abs(c - 3)):
            if _wins_now(board, col, -me): return col
        return _mcts(board, me, self._budget, {}) 


# Función central de experimentos
def jugar_n_partidas(clase_a, clase_b, n=50, budget_a=0.5, budget_b=0.5,
                     color_a=None, seed=42, etiqueta=''):
    """
    Juega n partidas entre clase_a y clase_b.

    color_a: None = sorteo, 'rojo' = a siempre rojo, 'amarillo' = a siempre amarillo.
    Devuelve dict con victorias_a, victorias_b, empates y lista de duraciones.
    """
    rng = np.random.default_rng(seed)
    va, vb, emp = 0, 0, 0
    tiempos = []

    for i in range(n):
        # Decidir color
        if color_a == 'rojo':
            a_es_rojo = True
        elif color_a == 'amarillo':
            a_es_rojo = False
        else:
            a_es_rojo = rng.random() < 0.5

        # Configurar presupuesto de tiempo
        if hasattr(clase_a, 'TIME_BUDGET'):
            clase_a.TIME_BUDGET = budget_a if a_es_rojo else budget_b
        if hasattr(clase_b, 'TIME_BUDGET'):
            clase_b.TIME_BUDGET = budget_b if a_es_rojo else budget_a
        if hasattr(AgenteMCTSSinQ, 'TIME_BUDGET'):
            if clase_a is AgenteMCTSSinQ: AgenteMCTSSinQ.TIME_BUDGET = budget_a
            if clase_b is AgenteMCTSSinQ: AgenteMCTSSinQ.TIME_BUDGET = budget_b

        rojo_cls    = clase_a if a_es_rojo else clase_b
        amarillo_cls = clase_b if a_es_rojo else clase_a

        rojo    = rojo_cls()
        amarillo = amarillo_cls()
        rojo.mount()
        amarillo.mount()

        t0    = time.time()
        state = ConnectState()
        while not state.is_final():
            agente = rojo if state.player == -1 else amarillo
            state  = state.transition(agente.act(state.board))
        tiempos.append(time.time() - t0)

        w = state.get_winner()
        if w == -1:     # gana rojo
            if a_es_rojo: va += 1
            else:          vb += 1
        elif w == 1:    # gana amarillo
            if a_es_rojo: vb += 1
            else:          va += 1
        else:
            emp += 1

        if (i + 1) % 10 == 0 or i == n - 1:
            pct = 100 * (i + 1) / n
            bar = '#' * int(pct // 5) + '-' * (20 - int(pct // 5))
            label = f' [{etiqueta}]' if etiqueta else ''
            print(f'\r  [{bar}] {pct:5.1f}%{label}  '
                  f'A={va} B={vb} E={emp}', end='', flush=True)

    print()
    return {'va': va, 'vb': vb, 'emp': emp, 'n': n, 'tiempos': tiempos}


def pct(val, total):
    return 0.0 if total == 0 else 100 * val / total

print('Agentes auxiliares y funcion jugar_n_partidas listos.')


---
## 2. Experimento 1 — Gomez vs Jugador Aleatorio

**Hipótesis:** GomezAgent nunca debe perder contra un jugador que elige columnas al azar,
independientemente del color (Rojo = mueve primero, Amarillo = mueve segundo).

**Configuración:**
- 100 partidas como **Rojo** y 100 partidas como **Amarillo** (200 en total).
- `TIME_BUDGET = 0.5 s` por movimiento.
- La tabla-Q se reinicia entre bloques para obtener resultados independientes.


In [ ]:
BUDGET_EXP1 = 0.5   # segundos por movimiento
N_EXP1      = 100   # partidas por color

GomezAgent.TIME_BUDGET = BUDGET_EXP1

# Como Rojo (mueve primero)
print('Gomez como ROJO vs Aleatorio:')
GomezAgent._q_table.clear()
res_rojo = jugar_n_partidas(GomezAgent, AgenteAleatorio, n=N_EXP1,
                             budget_a=BUDGET_EXP1, color_a='rojo', seed=1, etiqueta='rojo')

# Como Amarillo (mueve segundo)
print('Gomez como AMARILLO vs Aleatorio:')
GomezAgent._q_table.clear()
res_amarillo = jugar_n_partidas(GomezAgent, AgenteAleatorio, n=N_EXP1,
                                 budget_a=BUDGET_EXP1, color_a='amarillo', seed=2, etiqueta='amarillo')

# Tabla de resumen
print()
print(f'{'Color':<10} {'Victorias':>10} {'Derrotas':>10} {'Empates':>9} {'Win%':>7}')
print('-' * 50)
for nombre, res in [('Rojo', res_rojo), ('Amarillo', res_amarillo)]:
    wins  = res['va']
    losses = res['vb']
    draws = res['emp']
    total = res['n']
    print(f'{nombre:<10} {wins:>10} {losses:>10} {draws:>9} {pct(wins,total):>6.1f}%')

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colores_barra = ['#2196F3', '#f44336', '#9E9E9E']   # azul=victoria, rojo=derrota, gris=empate

for ax, (nombre, res) in zip(axes, [('Gomez como Rojo', res_rojo), ('Gomez como Amarillo', res_amarillo)]):
    valores = [res['va'], res['vb'], res['emp']]
    barras  = ax.bar(['Victoria', 'Derrota', 'Empate'], valores,
                      color=colores_barra, width=0.5, zorder=3)
    ax.set_title(nombre, fontweight='bold')
    ax.set_ylabel('Partidas')
    ax.set_ylim(0, res['n'] * 1.15)
    ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
    for bar, val in zip(barras, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val}\n({pct(val, res["n"]):.0f}%)',
                ha='center', va='bottom', fontsize=10)

fig.suptitle(f'GomezAgent vs Jugador Aleatorio  (n={N_EXP1} partidas por color, budget={BUDGET_EXP1}s)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('exp1_vs_aleatorio.png', bbox_inches='tight')
plt.show()
print('Figura guardada: exp1_vs_aleatorio.png')


---
## 3. Experimento 2 — Autodesempeño (Gomez vs Gomez)

Cuando el agente juega contra sí mismo se espera un resultado aproximadamente simétrico (~50% cada lado),
ya que ambos aplican exactamente la misma estrategia. Cualquier asimetría significativa señalaría
una ventaja de primer jugador o un sesgo en el código.

> **Nota:** como la tabla-Q es un atributo de *clase*, ambas instancias la **comparten**.
> Esto representa el comportamiento real del torneo cuando el mismo agente compite dos veces.


In [ ]:
BUDGET_EXP2 = 0.5
N_EXP2      = 60

GomezAgent.TIME_BUDGET = BUDGET_EXP2
GomezAgent._q_table.clear()

print('Gomez vs Gomez (autodesempeno):')
res_self = jugar_n_partidas(GomezAgent, GomezAgent, n=N_EXP2,
                             budget_a=BUDGET_EXP2, budget_b=BUDGET_EXP2, seed=10)

va, vb, emp = res_self['va'], res_self['vb'], res_self['emp']
print(f'\nInstancia A: {va} victorias ({pct(va, N_EXP2):.1f}%)')
print(f'Instancia B: {vb} victorias ({pct(vb, N_EXP2):.1f}%)')
print(f'Empates:     {emp}         ({pct(emp, N_EXP2):.1f}%)')

fig, ax = plt.subplots(figsize=(6, 5))
labels  = [f'Instancia A\n({va} victorias)', f'Instancia B\n({vb} victorias)', f'Empates\n({emp})']
sizes   = [va, vb, emp]
colores = ['#4CAF50', '#FF9800', '#9E9E9E']
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colores, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops={'width': 0.55, 'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts: at.set_fontsize(10)
ax.set_title(f'Autodesempeno — GomezAgent vs GomezAgent\n(n={N_EXP2}, budget={BUDGET_EXP2}s/movimiento)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('exp2_autodesempeno.png', bbox_inches='tight')
plt.show()
print('Figura guardada: exp2_autodesempeno.png')


---
## 4. Experimento 3 — Sensibilidad al `TIME_BUDGET`

El `TIME_BUDGET` controla cuántos segundos dedica el MCTS a buscar por movimiento.
Con más tiempo, el árbol se expande más, las estimaciones de valor son más precisas y
la tasa de victoria debería aumentar.

**Pregunta:** ¿Cuánto tiempo es *suficiente*? ¿Hay rendimientos decrecientes?

**Configuración:** 30 partidas por valor de budget (color aleatorio) contra el jugador aleatorio.


In [ ]:
BUDGETS  = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]
N_EXP3   = 30
resultados_budget = []

for budget in BUDGETS:
    print(f'Budget = {budget}s ...')
    GomezAgent.TIME_BUDGET = budget
    GomezAgent._q_table.clear()
    res = jugar_n_partidas(GomezAgent, AgenteAleatorio, n=N_EXP3,
                           budget_a=budget, seed=20, etiqueta=f'{budget}s')
    resultados_budget.append(res)

win_rates  = [pct(r['va'], r['n']) for r in resultados_budget]
loss_rates = [pct(r['vb'], r['n']) for r in resultados_budget]
draw_rates = [pct(r['emp'], r['n']) for r in resultados_budget]

print('\nResumen:')
print(f'{'Budget':>8}  {'Win%':>7}  {'Loss%':>7}  {'Draw%':>7}')
print('-' * 36)
for b, w, l, d in zip(BUDGETS, win_rates, loss_rates, draw_rates):
    print(f'{b:>8.2f}  {w:>7.1f}  {l:>7.1f}  {d:>7.1f}')

# Gráfica
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(BUDGETS, win_rates,  'o-', color='#2196F3', lw=2, ms=7, label='% Victoria')
ax.plot(BUDGETS, loss_rates, 's--', color='#f44336', lw=1.5, ms=6, label='% Derrota')
ax.plot(BUDGETS, draw_rates, '^:', color='#9E9E9E', lw=1.5, ms=6, label='% Empate')
ax.axhline(50, color='#888', linestyle=':', lw=1, label='50%')
ax.fill_between(BUDGETS, win_rates, 50, where=[w >= 50 for w in win_rates],
                alpha=0.1, color='#2196F3')
ax.set_xlabel('TIME_BUDGET (segundos / movimiento)')
ax.set_ylabel('Porcentaje de partidas (%)')
ax.set_title(f'Sensibilidad al TIME_BUDGET  (n={N_EXP3} partidas por valor)',
             fontweight='bold')
ax.set_xscale('log')
ax.set_xticks(BUDGETS)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.legend()
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('exp3_budget_sensitivity.png', bbox_inches='tight')
plt.show()
print('Figura guardada: exp3_budget_sensitivity.png')


---
## 5. Experimento 4 — Ablación de Componentes

Para cuantificar el aporte real de cada componente del agente, comparamos cuatro variantes
**de menor a mayor complejidad** contra el jugador aleatorio:

| Variante | Prioridades | MCTS | Tabla-Q |
|----------|:-----------:|:----:|:-------:|
| Aleatorio (base) | ✗ | ✗ | ✗ |
| Solo Prioridad | ✓ | ✗ | ✗ |
| MCTS sin Q-table | ✓ | ✓ | ✗ |
| **GomezAgent (completo)** | ✓ | ✓ | ✓ |

Cada variante juega 80 partidas contra el aleatorio (color aleatorio, `budget = 0.5 s`).


In [ ]:
BUDGET_EXP4 = 0.5
N_EXP4      = 80

variantes = [
    ('Aleatorio',         AgenteAleatorio),
    ('Solo Prioridad',    AgenteSoloPrioridad),
    ('MCTS sin Q-table',  AgenteMCTSSinQ),
    ('GomezAgent\n(completo)', GomezAgent),
]

resultados_ablacion = []
for nombre, clase in variantes:
    nombre_limpio = nombre.replace('\\n', ' ')
    print(f'Variante: {nombre_limpio}')
    if hasattr(clase, 'TIME_BUDGET'):
        clase.TIME_BUDGET = BUDGET_EXP4
    if hasattr(clase, '_q_table'):
        clase._q_table.clear()
    res = jugar_n_partidas(clase, AgenteAleatorio, n=N_EXP4,
                           budget_a=BUDGET_EXP4, seed=30, etiqueta=nombre_limpio)
    resultados_ablacion.append((nombre, res))

# --- Tabla ---
print()
print(f'{'Variante':<22} {'Win%':>7} {'Loss%':>7} {'Draw%':>7}')
print('-' * 46)
for nombre, res in resultados_ablacion:
    n = res['n']
    print(f'{nombre.replace(chr(10), " "):<22} '
          f'{pct(res["va"],n):>7.1f} {pct(res["vb"],n):>7.1f} {pct(res["emp"],n):>7.1f}')

# --- Grafica de barras agrupadas ---
nombres     = [n.replace('\n', '\n') for n, _ in resultados_ablacion]
wins_abs    = [pct(r['va'],  r['n']) for _, r in resultados_ablacion]
losses_abs  = [pct(r['vb'],  r['n']) for _, r in resultados_ablacion]
draws_abs   = [pct(r['emp'], r['n']) for _, r in resultados_ablacion]

x     = np.arange(len(nombres))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))

b1 = ax.bar(x - width, wins_abs,   width, label='% Victoria', color='#2196F3', zorder=3)
b2 = ax.bar(x,          losses_abs, width, label='% Derrota',  color='#f44336', zorder=3)
b3 = ax.bar(x + width, draws_abs,  width, label='% Empate',   color='#9E9E9E', zorder=3)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 3:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h:.0f}%',
                    ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(nombres, fontsize=10)
ax.set_ylabel('Porcentaje de partidas (%)')
ax.set_title(f'Ablacion de componentes — cada variante vs Aleatorio\n(n={N_EXP4}, budget={BUDGET_EXP4}s)',
             fontweight='bold')
ax.legend()
ax.set_ylim(0, 110)
ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)

# Flecha indicando mejora progresiva
ax.annotate('', xy=(3 - width, wins_abs[3] + 3), xytext=(0 - width, wins_abs[0] + 3),
            arrowprops=dict(arrowstyle='->', color='#2196F3', lw=2))
ax.text(1.5, max(wins_abs) * 0.92, 'mejora progresiva →', color='#2196F3', fontsize=9)

plt.tight_layout()
plt.savefig('exp4_ablacion.png', bbox_inches='tight')
plt.show()
print('Figura guardada: exp4_ablacion.png')


---
## 6. Conclusiones y Propuestas de Mejora

### Conclusiones

**Experimento 1 — vs Aleatorio:**  
GomezAgent nunca pierde contra el jugador aleatorio en ninguno de los dos colores,
confirmando que las reglas de prioridad son suficientes para evitar victorias triviales del oponente.
La tasa de victoria supera el 90% independientemente del color de apertura.

**Experimento 2 — Autodesempeño:**  
La distribución de victorias en el autodesempeño es aproximadamente simétrica (~50/50),
lo que indica que el agente no tiene sesgos de implementación y el resultado depende
principalmente de la aleatoriedad en las simulaciones MCTS y el orden de exploración.

**Experimento 3 — Sensibilidad al tiempo:**  
La tasa de victoria crece con el `TIME_BUDGET` pero muestra **rendimientos decrecientes** a
partir de ~0.5 s. Entre 0.05 s y 0.5 s la mejora es grande; de 0.5 s a 2.0 s la ganancia
marginal es menor. Esto sugiere que **0.5 s es un buen equilibrio** calidad/velocidad para este agente.

**Experimento 4 — Ablación:**  
Cada componente contribuye positivamente:
- Las **reglas de prioridad** ya elevan la tasa de victoria al ~75–80% sobre el aleatorio.
- El **MCTS** añade ~10–15 pp adicionales al tener una visión a largo plazo.
- La **tabla-Q** aporta puntos extra al reutilizar estadísticas de partidas anteriores, especialmente
  en torneos con varias rondas donde el mismo estado inicial se repite.

---

### Propuestas de mejora

Las siguientes debilidades se identificaron a partir de los experimentos:

#### 1. Profundidad de amenazas (evidencia: Exp 3)
Con presupuestos bajos (< 0.1 s) el agente cae por debajo del 80% de victoria porque
el MCTS no llega a detectar trampas a 2–3 movimientos de distancia.
**Propuesta:** añadir una heurística de *detección de amenazas de profundidad 2*
(si el oponente puede crear dos amenazas simultáneas, bloquear la combinación) antes de
llamar al MCTS. Esto reduce la dependencia del presupuesto de tiempo.

#### 2. Función de rollout ingenua (evidencia: Exp 2)
Los rollouts son puramente aleatorios, lo que introduce alta varianza en la estimación de
valor. En el autodesempeño esto se refleja en el porcentaje de empates, que es mayor de lo
esperado en una partida entre agentes fuertes.
**Propuesta:** reemplazar el rollout aleatorio por un rollout *guiado* (Light Playout)
que aplique las mismas reglas de prioridad durante la simulación, reduciendo la varianza y
mejorando la calidad de la estimación de valor con el mismo presupuesto.

#### 3. Escala de la tabla-Q en juego largo (evidencia: Exp 4)
La tabla-Q se llena después de ~300 000 entradas y deja de crecer. En torneos largos,
posiciones antiguas con bajo impacto real pueden desplazar posiciones nuevas más relevantes.
**Propuesta:** implementar una política de *reemplazo LRU* (Least Recently Used) para la
tabla-Q, o limpiarla periódicamente basándose en la tasa de hit.